<a href="https://colab.research.google.com/github/busycaesar/Finetune_LoRA/blob/Master/OpenWeightModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hugging Face Login

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## Load the data

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("sgoel9/paul_graham_essays", split="train")

print(raw_dataset[0])

## Structure the data

In [ ]:
from datasets import Dataset

# Convert list into a Dataset object.
# It is needed so we can split it into train/test parts later.
dataset = Dataset.from_list([
    {
        "text": f"Title: {row['title']}\n\nEssay:\n{row['text']}"
    }
    for row in raw_dataset
])

print(dataset[1])

## Split the data into test and train sets

In [ ]:
# seed fixes the random split so it's reproducible
dataset = dataset.train_test_split(test_size=0.1, seed = 42)

## Load and quantize the base model

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct" # "meta-llama/Llama-3.2-3B-Instruct"

# load the tokenizer that matches the base model, using the HF token since it's gated
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)

# Llama has no pad token by default, so reuse the end-of-sequence token for padding
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
!pip install -q bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Sets up 4 bit loading which makes QLoRA fit on a T4.
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# Load the model from pretrained weights in quantized form and load it on a GPU.
# model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL,
#     quantization_config=bnb_config,
#     device_map="auto",
#     token=hf_token,
# )

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
    device_map="auto",
    token=hf_token,
)

## Configure LoRA

In [ ]:
!pip install -q peft -U torchao

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
!pip install -q trl

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./pg-essay-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    dataset_text_field="text",
    max_length=1024,
    packing=False,
    report_to="none",
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

In [ ]:
trainer.save_model("./pg-essay-lora-final")

model.push_to_hub("busycaesar/pg-essay-lora")

In [ ]:
model.eval()
model.config.use_cache = True

prompt = "Title: How to Start a Startup\n\nEssay:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with model.disable_adapter():
    with torch.no_grad():
        base_out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
    print("===== BASE =====")
    print(tokenizer.decode(base_out[0], skip_special_tokens=True))

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)

print(tokenizer.decode(output[0], skip_special_tokens=True))